[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/xla/lab-x5-mock-pjrt-plugin.ipynb)

# LAB·X5 · The mock device

**Hardware:** any machine. Everything below runs on CPU.

Chapter 14 read the PJRT interface as a list of promises. This lab keeps them. You will write a PJRT plugin in plain C, one platform, one device, one memory space, and hand it to an unmodified JAX. The device you build compiles programs and holds buffers; the one thing it refuses to do is execute. By the end, `jax.devices()` prints hardware that does not exist, and every error message you see is one you wrote.

Nothing here requires a GPU, a TPU, or the XLA build system. A C compiler and one header are the whole toolchain.

In [ ]:
# The track's pinned JAX. The C header we fetch next comes from the exact
# XLA commit this jaxlib was built from, so the two sides of the ABI agree.
%pip install -q "jax==0.4.38" "jaxlib==0.4.38"
import jax
print("jax", jax.__version__)

## One header is the whole SDK

A PJRT plugin needs exactly one header: `xla/pjrt/c/pjrt_c_api.h`. No XLA build, no protobufs, no C++. But the header must speak the same C API version as the jaxlib on the other side. jaxlib 0.4.38 was built from XLA commit `20a4825`, where the API is version 0.58; today's main is past 0.114 and has changed the shape of the error struct, so a main-branch header would not even compile against this lab's plugin.

That pin is chapter 14's versioning story made concrete: the framework and the plugin each report the version they were compiled against, and the header is where that number comes from.

**your prediction:** roughly how many C functions do you think a plugin must fill in before `jax.devices()` returns without crashing?

In [ ]:
import pathlib, urllib.request

XLA_COMMIT = "20a482597b7dd3067b26ca382b88084ee5a21cf7"  # pinned by jax v0.4.38
hdr_dir = pathlib.Path("include/xla/pjrt/c")
hdr_dir.mkdir(parents=True, exist_ok=True)
url = f"https://raw.githubusercontent.com/openxla/xla/{XLA_COMMIT}/xla/pjrt/c/pjrt_c_api.h"
urllib.request.urlretrieve(url, hdr_dir / "pjrt_c_api.h")

for line in (hdr_dir / "pjrt_c_api.h").read_text().splitlines():
    if "PJRT_API_MAJOR" in line or "PJRT_API_MINOR" in line:
        print(line)

## What we are about to implement

The source below is long but has only seven ideas, and each one is a section of chapter 14 turned into code:

1. **Errors.** `PJRT_Error` is opaque in the header; the plugin owns its layout. Ours is an error code and a message buffer, and the framework asks for both through `PJRT_Error_Message` and `PJRT_Error_GetCode`.
2. **Identity.** Client, device, device description, memory: one static instance of each. The accessor functions (`PJRT_Client_Devices`, `PJRT_DeviceDescription_Kind`, and their siblings) just hand back pointers to those statics. This is the state a real client builds at construction; ours is built at link time.
3. **Compile.** `PJRT_Client_Compile` receives the program JAX lowered, StableHLO as MLIR bytecode, and stores the bytes verbatim. Compilation, in this plugin, is the identity function. `PJRT_Executable_OptimizedProgram` hands the same bytes back when asked.
4. **Buffers.** A `PjRtBuffer` is memory on one device, and this device's memory is `malloc`. `PJRT_Client_BufferFromHostBuffer` copies the host array in; `PJRT_Buffer_ToHostBuffer` copies it back out. The interface never says what a device is. It only says who owns the bytes.
5. **Events.** Everything this plugin does finishes before the call returns, so every event is born ready and `PJRT_Event_OnReady` fires its callback immediately. The async machinery is all still there; it just never waits.
6. **The refusal.** `PJRT_LoadedExecutable_Execute` returns UNIMPLEMENTED with a message you will recognize later. This is the one promise we decline on purpose.
7. **The table.** `GetPjrtApi()` fills a `PJRT_Api` struct with function pointers and returns it. That one exported symbol is the entire plugin ABI.

In [ ]:
mock_pjrt_c = r'''// mock_pjrt.c: the smallest PJRT plugin that a real JAX will talk to.
//
// One platform ("mock"), one device, one memory space. Compile accepts the
// program and stores it. Execute refuses, on purpose. Everything lives in
// host memory; there is no hardware and no compiler behind this table.
//
// Build:  cc -shared -fPIC -I <xla-src-root> mock_pjrt.c -o mock_pjrt.so

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#include "xla/pjrt/c/pjrt_c_api.h"

// ---------------------------------------------------------------- errors ----
// The C API reports failure by returning a PJRT_Error*; NULL means success.
// The framework asks a failed call for its message and code, then destroys it.

struct PJRT_Error {
  PJRT_Error_Code code;
  char message[512];
};

static PJRT_Error* make_error(PJRT_Error_Code code, const char* msg) {
  PJRT_Error* err = (PJRT_Error*)malloc(sizeof(PJRT_Error));
  err->code = code;
  snprintf(err->message, sizeof(err->message), "%s", msg);
  return err;
}

static void Error_Destroy(PJRT_Error_Destroy_Args* args) {
  free(args->error);
}

static void Error_Message(PJRT_Error_Message_Args* args) {
  args->message = args->error->message;
  args->message_size = strlen(args->error->message);
}

static PJRT_Error* Error_GetCode(PJRT_Error_GetCode_Args* args) {
  args->code = args->error->code;
  return NULL;
}

#define UNIMPLEMENTED(name)                                        \
  make_error(PJRT_Error_Code_UNIMPLEMENTED,                        \
             "mock PJRT plugin: " name " is not implemented; this " \
             "device compiles but does not execute")

// ---------------------------------------------------------------- objects ---
// The header only forward-declares these structs; the plugin owns their
// layout. One static instance each: this plugin is one device, full stop.

struct PJRT_DeviceDescription {
  int id;
};
struct PJRT_Memory {
  int id;
};
struct PJRT_Device {
  struct PJRT_DeviceDescription* description;
  struct PJRT_Memory* memory;
};
struct PJRT_Client {
  struct PJRT_Device* device;
  struct PJRT_Memory* memory;
};
struct PJRT_Executable {
  char* program;        // the bytes Compile was handed, verbatim
  size_t program_size;
  char format[16];      // "mlir" or "hlo", whatever the caller said it was
  int ref_count;
};
struct PJRT_LoadedExecutable {
  struct PJRT_Executable* executable;
  struct PJRT_Client* client;
};

static struct PJRT_DeviceDescription mock_description = {0};
static struct PJRT_Memory mock_memory = {0};
static struct PJRT_Device mock_device = {&mock_description, &mock_memory};
static struct PJRT_Client mock_client = {&mock_device, &mock_memory};
static PJRT_Device* mock_devices[1] = {&mock_device};
static PJRT_Memory* mock_memories[1] = {&mock_memory};

// ----------------------------------------------------------------- plugin ---

static PJRT_Error* Plugin_Initialize(PJRT_Plugin_Initialize_Args* args) {
  (void)args;
  return NULL;
}

static PJRT_Error* Plugin_Attributes(PJRT_Plugin_Attributes_Args* args) {
  args->num_attributes = 0;
  args->attributes = NULL;
  return NULL;
}

// ----------------------------------------------------------------- client ---

static PJRT_Error* Client_Create(PJRT_Client_Create_Args* args) {
  args->client = &mock_client;
  return NULL;
}

static PJRT_Error* Client_Destroy(PJRT_Client_Destroy_Args* args) {
  (void)args;  // everything is static; nothing to free
  return NULL;
}

static PJRT_Error* Client_PlatformName(PJRT_Client_PlatformName_Args* args) {
  args->platform_name = "mock";
  args->platform_name_size = 4;
  return NULL;
}

static PJRT_Error* Client_ProcessIndex(PJRT_Client_ProcessIndex_Args* args) {
  args->process_index = 0;
  return NULL;
}

static PJRT_Error* Client_PlatformVersion(
    PJRT_Client_PlatformVersion_Args* args) {
  args->platform_version = "mock 0.1";
  args->platform_version_size = 8;
  return NULL;
}

static PJRT_Error* Client_Devices(PJRT_Client_Devices_Args* args) {
  args->devices = mock_devices;
  args->num_devices = 1;
  return NULL;
}

static PJRT_Error* Client_AddressableDevices(
    PJRT_Client_AddressableDevices_Args* args) {
  args->addressable_devices = mock_devices;
  args->num_addressable_devices = 1;
  return NULL;
}

static PJRT_Error* Client_LookupDevice(PJRT_Client_LookupDevice_Args* args) {
  if (args->id != 0)
    return make_error(PJRT_Error_Code_NOT_FOUND, "mock has exactly device 0");
  args->device = &mock_device;
  return NULL;
}

static PJRT_Error* Client_LookupAddressableDevice(
    PJRT_Client_LookupAddressableDevice_Args* args) {
  if (args->local_hardware_id != 0)
    return make_error(PJRT_Error_Code_NOT_FOUND, "mock has exactly device 0");
  args->addressable_device = &mock_device;
  return NULL;
}

static PJRT_Error* Client_AddressableMemories(
    PJRT_Client_AddressableMemories_Args* args) {
  args->addressable_memories = mock_memories;
  args->num_addressable_memories = 1;
  return NULL;
}

static PJRT_Error* Client_DefaultDeviceAssignment(
    PJRT_Client_DefaultDeviceAssignment_Args* args) {
  for (size_t i = 0; i < args->default_assignment_size; ++i) {
    args->default_assignment[i] = 0;
  }
  return NULL;
}

static PJRT_Error* Client_TopologyDescription(
    PJRT_Client_TopologyDescription_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Client_TopologyDescription");
}

// ----------------------------------------------------- device description ---

static PJRT_Error* DeviceDescription_Id(PJRT_DeviceDescription_Id_Args* args) {
  args->id = args->device_description->id;
  return NULL;
}

static PJRT_Error* DeviceDescription_ProcessIndex(
    PJRT_DeviceDescription_ProcessIndex_Args* args) {
  args->process_index = 0;
  return NULL;
}

static PJRT_Error* DeviceDescription_Attributes(
    PJRT_DeviceDescription_Attributes_Args* args) {
  args->num_attributes = 0;
  args->attributes = NULL;
  return NULL;
}

static PJRT_Error* DeviceDescription_Kind(
    PJRT_DeviceDescription_Kind_Args* args) {
  args->device_kind = "mock";
  args->device_kind_size = 4;
  return NULL;
}

static PJRT_Error* DeviceDescription_DebugString(
    PJRT_DeviceDescription_DebugString_Args* args) {
  args->debug_string = "MockDevice(id=0)";
  args->debug_string_size = strlen("MockDevice(id=0)");
  return NULL;
}

static PJRT_Error* DeviceDescription_ToString(
    PJRT_DeviceDescription_ToString_Args* args) {
  args->to_string = "mock:0";
  args->to_string_size = 6;
  return NULL;
}

// ----------------------------------------------------------------- device ---

static PJRT_Error* Device_GetDescription(PJRT_Device_GetDescription_Args* args) {
  args->device_description = args->device->description;
  return NULL;
}

static PJRT_Error* Device_IsAddressable(PJRT_Device_IsAddressable_Args* args) {
  args->is_addressable = true;
  return NULL;
}

static PJRT_Error* Device_LocalHardwareId(
    PJRT_Device_LocalHardwareId_Args* args) {
  args->local_hardware_id = 0;
  return NULL;
}

static PJRT_Error* Device_AddressableMemories(
    PJRT_Device_AddressableMemories_Args* args) {
  args->memories = mock_memories;
  args->num_memories = 1;
  return NULL;
}

static PJRT_Error* Device_DefaultMemory(PJRT_Device_DefaultMemory_Args* args) {
  args->memory = &mock_memory;
  return NULL;
}

static PJRT_Error* Device_MemoryStats(PJRT_Device_MemoryStats_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Device_MemoryStats");
}

// ----------------------------------------------------------------- memory ---

static PJRT_Error* Memory_Id(PJRT_Memory_Id_Args* args) {
  args->id = 0;
  return NULL;
}

static PJRT_Error* Memory_Kind(PJRT_Memory_Kind_Args* args) {
  args->kind = "mock_host";
  args->kind_size = strlen("mock_host");
  return NULL;
}

static PJRT_Error* Memory_KindId(PJRT_Memory_Kind_Id_Args* args) {
  args->kind_id = 0;
  return NULL;
}

static PJRT_Error* Memory_DebugString(PJRT_Memory_DebugString_Args* args) {
  args->debug_string = "MockMemory(mock_host)";
  args->debug_string_size = strlen("MockMemory(mock_host)");
  return NULL;
}

static PJRT_Error* Memory_ToString(PJRT_Memory_ToString_Args* args) {
  args->to_string = "mock_host:0";
  args->to_string_size = strlen("mock_host:0");
  return NULL;
}

static PJRT_Error* Memory_AddressableByDevices(
    PJRT_Memory_AddressableByDevices_Args* args) {
  args->devices = mock_devices;
  args->num_devices = 1;
  return NULL;
}

// ---------------------------------------------------------------- compile ---
// The one promise this plugin keeps for real: accept the program. The bytes
// JAX sends are StableHLO as MLIR bytecode; we store them verbatim and hand
// them back if asked for the "optimized" program. Compilation, here, is the
// identity function. Everything the interface requires still works.

static PJRT_Error* Client_Compile(PJRT_Client_Compile_Args* args) {
  struct PJRT_Executable* exe =
      (struct PJRT_Executable*)malloc(sizeof(struct PJRT_Executable));
  exe->program_size = args->program->code_size;
  exe->program = (char*)malloc(exe->program_size);
  memcpy(exe->program, args->program->code, exe->program_size);
  snprintf(exe->format, sizeof(exe->format), "%.*s",
           (int)args->program->format_size, args->program->format);
  exe->ref_count = 1;

  struct PJRT_LoadedExecutable* loaded =
      (struct PJRT_LoadedExecutable*)malloc(
          sizeof(struct PJRT_LoadedExecutable));
  loaded->executable = exe;
  loaded->client = args->client;
  args->executable = loaded;
  return NULL;
}

static PJRT_Error* Executable_Destroy(PJRT_Executable_Destroy_Args* args) {
  struct PJRT_Executable* exe = args->executable;
  if (--exe->ref_count == 0) {
    free(exe->program);
    free(exe);
  }
  return NULL;
}

static PJRT_Error* LoadedExecutable_Destroy(
    PJRT_LoadedExecutable_Destroy_Args* args) {
  struct PJRT_Executable* exe = args->executable->executable;
  if (--exe->ref_count == 0) {
    free(exe->program);
    free(exe);
  }
  free(args->executable);
  return NULL;
}

static PJRT_Error* LoadedExecutable_GetExecutable(
    PJRT_LoadedExecutable_GetExecutable_Args* args) {
  args->loaded_executable->executable->ref_count++;
  args->executable = args->loaded_executable->executable;
  return NULL;
}

static PJRT_Error* Executable_Name(PJRT_Executable_Name_Args* args) {
  args->executable_name = "mock_executable";
  args->executable_name_size = strlen("mock_executable");
  return NULL;
}

static PJRT_Error* Executable_NumReplicas(
    PJRT_Executable_NumReplicas_Args* args) {
  args->num_replicas = 1;
  return NULL;
}

static PJRT_Error* Executable_NumPartitions(
    PJRT_Executable_NumPartitions_Args* args) {
  args->num_partitions = 1;
  return NULL;
}

static PJRT_Error* LoadedExecutable_AddressableDevices(
    PJRT_LoadedExecutable_AddressableDevices_Args* args) {
  args->addressable_devices = mock_devices;
  args->num_addressable_devices = 1;
  return NULL;
}

static PJRT_Error* LoadedExecutable_Delete(
    PJRT_LoadedExecutable_Delete_Args* args) {
  (void)args;
  return NULL;
}

static PJRT_Error* LoadedExecutable_IsDeleted(
    PJRT_LoadedExecutable_IsDeleted_Args* args) {
  args->is_deleted = false;
  return NULL;
}

static PJRT_Error* Executable_OptimizedProgram(
    PJRT_Executable_OptimizedProgram_Args* args) {
  struct PJRT_Executable* exe = args->executable;
  PJRT_Program* out = args->program;
  out->format = exe->format;
  out->format_size = strlen(exe->format);
  if (out->code == NULL) {
    // First call: report the size. Second call: copy the bytes.
    out->code_size = exe->program_size;
    return NULL;
  }
  memcpy(out->code, exe->program, exe->program_size);
  return NULL;
}

static PJRT_Error* Executable_NumOutputs(PJRT_Executable_NumOutputs_Args* args) {
  args->num_outputs = 1;
  return NULL;
}

static PJRT_Error* Executable_SizeOfGeneratedCodeInBytes(
    PJRT_Executable_SizeOfGeneratedCodeInBytes_Args* args) {
  args->size_in_bytes = (int64_t)args->executable->program_size;
  return NULL;
}

static PJRT_Error* Executable_GetCostAnalysis(
    PJRT_Executable_GetCostAnalysis_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_GetCostAnalysis");
}

static PJRT_Error* Executable_OutputMemoryKinds(
    PJRT_Executable_OutputMemoryKinds_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_OutputMemoryKinds");
}

static PJRT_Error* Executable_OutputElementTypes(
    PJRT_Executable_OutputElementTypes_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_OutputElementTypes");
}

static PJRT_Error* Executable_OutputDimensions(
    PJRT_Executable_OutputDimensions_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Executable_OutputDimensions");
}

static PJRT_Error* Executable_Fingerprint(
    PJRT_Executable_Fingerprint_Args* args) {
  args->executable_fingerprint = "mock-fingerprint";
  args->executable_fingerprint_size = strlen("mock-fingerprint");
  return NULL;
}

// ----------------------------------------------------------------- events ---
// Everything this plugin does finishes before the call returns, so every
// event is born ready. The framework still expects the full event protocol.

struct PJRT_Event {
  int dummy;
};

static PJRT_Error* Event_Destroy(PJRT_Event_Destroy_Args* args) {
  free(args->event);
  return NULL;
}

static PJRT_Error* Event_IsReady(PJRT_Event_IsReady_Args* args) {
  args->is_ready = true;
  return NULL;
}

static PJRT_Error* Event_Error(PJRT_Event_Error_Args* args) {
  (void)args;
  return NULL;  // a ready event with no error
}

static PJRT_Error* Event_Await(PJRT_Event_Await_Args* args) {
  (void)args;
  return NULL;
}

static PJRT_Error* Event_OnReady(PJRT_Event_OnReady_Args* args) {
  args->callback(NULL, args->user_arg);  // already ready: fire immediately
  return NULL;
}

static struct PJRT_Event* make_ready_event(void) {
  return (struct PJRT_Event*)calloc(1, sizeof(struct PJRT_Event));
}

// ---------------------------------------------------------------- buffers ---
// A PjRtBuffer is memory on one device. This device's memory is plain host
// memory the plugin mallocs, which is the honest version of the lesson: the
// interface never says what a device is, only who owns the bytes.

#define MOCK_MAX_DIMS 16

struct PJRT_Buffer {
  char* data;
  size_t size_bytes;
  PJRT_Buffer_Type type;
  int64_t dims[MOCK_MAX_DIMS];
  int64_t minor_to_major[MOCK_MAX_DIMS];
  size_t num_dims;
  bool deleted;
};

static const size_t mock_elem_size[] = {
    [PJRT_Buffer_Type_PRED] = 1,  [PJRT_Buffer_Type_S8] = 1,
    [PJRT_Buffer_Type_S16] = 2,   [PJRT_Buffer_Type_S32] = 4,
    [PJRT_Buffer_Type_S64] = 8,   [PJRT_Buffer_Type_U8] = 1,
    [PJRT_Buffer_Type_U16] = 2,   [PJRT_Buffer_Type_U32] = 4,
    [PJRT_Buffer_Type_U64] = 8,   [PJRT_Buffer_Type_F16] = 2,
    [PJRT_Buffer_Type_F32] = 4,   [PJRT_Buffer_Type_F64] = 8,
    [PJRT_Buffer_Type_BF16] = 2,  [PJRT_Buffer_Type_C64] = 8,
    [PJRT_Buffer_Type_C128] = 16,
};

static PJRT_Error* Client_BufferFromHostBuffer(
    PJRT_Client_BufferFromHostBuffer_Args* args) {
  if (args->num_dims > MOCK_MAX_DIMS)
    return make_error(PJRT_Error_Code_UNIMPLEMENTED,
                      "mock PJRT plugin: too many dimensions");
  if (args->type >= sizeof(mock_elem_size) / sizeof(mock_elem_size[0]) ||
      mock_elem_size[args->type] == 0)
    return make_error(PJRT_Error_Code_UNIMPLEMENTED,
                      "mock PJRT plugin: unsupported element type");
  if (args->byte_strides != NULL && args->num_byte_strides != 0) {
    // Accept only dense row-major strides; anything else stays unimplemented.
    int64_t expect = (int64_t)mock_elem_size[args->type];
    for (size_t i = args->num_dims; i > 0; --i) {
      if (args->byte_strides[i - 1] != expect)
        return UNIMPLEMENTED("non-row-major byte_strides");
      expect *= args->dims[i - 1];
    }
  }

  struct PJRT_Buffer* buf =
      (struct PJRT_Buffer*)calloc(1, sizeof(struct PJRT_Buffer));
  buf->type = args->type;
  buf->num_dims = args->num_dims;
  size_t n = mock_elem_size[args->type];
  for (size_t i = 0; i < args->num_dims; ++i) {
    buf->dims[i] = args->dims[i];
    buf->minor_to_major[i] = (int64_t)(args->num_dims - 1 - i);
    n *= (size_t)args->dims[i];
  }
  buf->size_bytes = n;
  buf->data = (char*)malloc(n);
  memcpy(buf->data, args->data, n);  // "transfer" to the device: one memcpy

  args->buffer = buf;
  args->done_with_host_buffer = make_ready_event();
  return NULL;
}

static PJRT_Error* Buffer_Destroy(PJRT_Buffer_Destroy_Args* args) {
  free(args->buffer->data);
  free(args->buffer);
  return NULL;
}

static PJRT_Error* Buffer_ElementType(PJRT_Buffer_ElementType_Args* args) {
  args->type = args->buffer->type;
  return NULL;
}

static PJRT_Error* Buffer_Dimensions(PJRT_Buffer_Dimensions_Args* args) {
  args->dims = args->buffer->dims;
  args->num_dims = args->buffer->num_dims;
  return NULL;
}

static PJRT_Error* Buffer_UnpaddedDimensions(
    PJRT_Buffer_UnpaddedDimensions_Args* args) {
  args->unpadded_dims = args->buffer->dims;
  args->num_dims = args->buffer->num_dims;
  return NULL;
}

static PJRT_Error* Buffer_DynamicDimensionIndices(
    PJRT_Buffer_DynamicDimensionIndices_Args* args) {
  args->dynamic_dim_indices = NULL;
  args->num_dynamic_dims = 0;
  return NULL;
}

static PJRT_Error* Buffer_GetMemoryLayout(
    PJRT_Buffer_GetMemoryLayout_Args* args) {
  args->layout.type = PJRT_Buffer_MemoryLayout_Type_Tiled;
  args->layout.tiled.minor_to_major = args->buffer->minor_to_major;
  args->layout.tiled.minor_to_major_size = args->buffer->num_dims;
  args->layout.tiled.tile_dims = NULL;
  args->layout.tiled.tile_dim_sizes = NULL;
  args->layout.tiled.num_tiles = 0;
  return NULL;
}

static PJRT_Error* Buffer_OnDeviceSizeInBytes(
    PJRT_Buffer_OnDeviceSizeInBytes_Args* args) {
  args->on_device_size_in_bytes = args->buffer->size_bytes;
  return NULL;
}

static PJRT_Error* Buffer_Device(PJRT_Buffer_Device_Args* args) {
  args->device = &mock_device;
  return NULL;
}

static PJRT_Error* Buffer_Memory(PJRT_Buffer_Memory_Args* args) {
  args->memory = &mock_memory;
  return NULL;
}

static PJRT_Error* Buffer_Delete(PJRT_Buffer_Delete_Args* args) {
  args->buffer->deleted = true;
  return NULL;
}

static PJRT_Error* Buffer_IsDeleted(PJRT_Buffer_IsDeleted_Args* args) {
  args->is_deleted = args->buffer->deleted;
  return NULL;
}

static PJRT_Error* Buffer_IsOnCpu(PJRT_Buffer_IsOnCpu_Args* args) {
  args->is_on_cpu = false;  // the polite fiction that makes it a device
  return NULL;
}

static PJRT_Error* Buffer_ReadyEvent(PJRT_Buffer_ReadyEvent_Args* args) {
  args->event = make_ready_event();
  return NULL;
}

static PJRT_Error* Buffer_ToHostBuffer(PJRT_Buffer_ToHostBuffer_Args* args) {
  struct PJRT_Buffer* buf = args->src;
  if (args->dst == NULL) {
    // First call: the framework asks how much space it needs.
    args->dst_size = buf->size_bytes;
    return NULL;
  }
  if (args->dst_size < buf->size_bytes)
    return make_error(PJRT_Error_Code_INVALID_ARGUMENT,
                      "mock PJRT plugin: destination too small");
  memcpy(args->dst, buf->data, buf->size_bytes);
  args->event = make_ready_event();
  return NULL;
}

static PJRT_Error* Buffer_CopyToDevice(PJRT_Buffer_CopyToDevice_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Buffer_CopyToDevice");
}

static PJRT_Error* Buffer_CopyToMemory(PJRT_Buffer_CopyToMemory_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_Buffer_CopyToMemory");
}

// ---------------------------------------------------------------- execute ---
// The refusal that makes this a mock. The error code and message travel the
// same path a real failure would: C error -> PjRtCApiClient -> Python.

static PJRT_Error* LoadedExecutable_Execute(
    PJRT_LoadedExecutable_Execute_Args* args) {
  (void)args;
  return UNIMPLEMENTED("PJRT_LoadedExecutable_Execute");
}

// ------------------------------------------------------------------ table ---

static PJRT_Api mock_api;

const PJRT_Api* GetPjrtApi(void) {
  memset(&mock_api, 0, sizeof(mock_api));
  mock_api.struct_size = PJRT_Api_STRUCT_SIZE;
  mock_api.pjrt_api_version.struct_size = PJRT_Api_Version_STRUCT_SIZE;
  mock_api.pjrt_api_version.major_version = PJRT_API_MAJOR;
  mock_api.pjrt_api_version.minor_version = PJRT_API_MINOR;

  mock_api.PJRT_Error_Destroy = Error_Destroy;
  mock_api.PJRT_Error_Message = Error_Message;
  mock_api.PJRT_Error_GetCode = Error_GetCode;

  mock_api.PJRT_Plugin_Initialize = Plugin_Initialize;
  mock_api.PJRT_Plugin_Attributes = Plugin_Attributes;

  mock_api.PJRT_Client_Create = Client_Create;
  mock_api.PJRT_Client_Destroy = Client_Destroy;
  mock_api.PJRT_Client_PlatformName = Client_PlatformName;
  mock_api.PJRT_Client_ProcessIndex = Client_ProcessIndex;
  mock_api.PJRT_Client_PlatformVersion = Client_PlatformVersion;
  mock_api.PJRT_Client_Devices = Client_Devices;
  mock_api.PJRT_Client_AddressableDevices = Client_AddressableDevices;
  mock_api.PJRT_Client_LookupDevice = Client_LookupDevice;
  mock_api.PJRT_Client_LookupAddressableDevice = Client_LookupAddressableDevice;
  mock_api.PJRT_Client_AddressableMemories = Client_AddressableMemories;
  mock_api.PJRT_Client_DefaultDeviceAssignment = Client_DefaultDeviceAssignment;
  mock_api.PJRT_Client_TopologyDescription = Client_TopologyDescription;
  mock_api.PJRT_Client_Compile = Client_Compile;
  mock_api.PJRT_Client_BufferFromHostBuffer = Client_BufferFromHostBuffer;

  mock_api.PJRT_DeviceDescription_Id = DeviceDescription_Id;
  mock_api.PJRT_DeviceDescription_ProcessIndex = DeviceDescription_ProcessIndex;
  mock_api.PJRT_DeviceDescription_Attributes = DeviceDescription_Attributes;
  mock_api.PJRT_DeviceDescription_Kind = DeviceDescription_Kind;
  mock_api.PJRT_DeviceDescription_DebugString = DeviceDescription_DebugString;
  mock_api.PJRT_DeviceDescription_ToString = DeviceDescription_ToString;

  mock_api.PJRT_Device_GetDescription = Device_GetDescription;
  mock_api.PJRT_Device_IsAddressable = Device_IsAddressable;
  mock_api.PJRT_Device_LocalHardwareId = Device_LocalHardwareId;
  mock_api.PJRT_Device_AddressableMemories = Device_AddressableMemories;
  mock_api.PJRT_Device_DefaultMemory = Device_DefaultMemory;
  mock_api.PJRT_Device_MemoryStats = Device_MemoryStats;

  mock_api.PJRT_Memory_Id = Memory_Id;
  mock_api.PJRT_Memory_Kind = Memory_Kind;
  mock_api.PJRT_Memory_Kind_Id = Memory_KindId;
  mock_api.PJRT_Memory_DebugString = Memory_DebugString;
  mock_api.PJRT_Memory_ToString = Memory_ToString;
  mock_api.PJRT_Memory_AddressableByDevices = Memory_AddressableByDevices;

  mock_api.PJRT_Executable_Destroy = Executable_Destroy;
  mock_api.PJRT_Executable_Name = Executable_Name;
  mock_api.PJRT_Executable_NumReplicas = Executable_NumReplicas;
  mock_api.PJRT_Executable_NumPartitions = Executable_NumPartitions;
  mock_api.PJRT_Executable_NumOutputs = Executable_NumOutputs;
  mock_api.PJRT_Executable_SizeOfGeneratedCodeInBytes =
      Executable_SizeOfGeneratedCodeInBytes;
  mock_api.PJRT_Executable_GetCostAnalysis = Executable_GetCostAnalysis;
  mock_api.PJRT_Executable_OutputMemoryKinds = Executable_OutputMemoryKinds;
  mock_api.PJRT_Executable_OutputElementTypes = Executable_OutputElementTypes;
  mock_api.PJRT_Executable_OutputDimensions = Executable_OutputDimensions;
  mock_api.PJRT_Executable_OptimizedProgram = Executable_OptimizedProgram;
  mock_api.PJRT_Executable_Fingerprint = Executable_Fingerprint;

  mock_api.PJRT_Event_Destroy = Event_Destroy;
  mock_api.PJRT_Event_IsReady = Event_IsReady;
  mock_api.PJRT_Event_Error = Event_Error;
  mock_api.PJRT_Event_Await = Event_Await;
  mock_api.PJRT_Event_OnReady = Event_OnReady;

  mock_api.PJRT_Buffer_Destroy = Buffer_Destroy;
  mock_api.PJRT_Buffer_ElementType = Buffer_ElementType;
  mock_api.PJRT_Buffer_Dimensions = Buffer_Dimensions;
  mock_api.PJRT_Buffer_UnpaddedDimensions = Buffer_UnpaddedDimensions;
  mock_api.PJRT_Buffer_DynamicDimensionIndices = Buffer_DynamicDimensionIndices;
  mock_api.PJRT_Buffer_GetMemoryLayout = Buffer_GetMemoryLayout;
  mock_api.PJRT_Buffer_OnDeviceSizeInBytes = Buffer_OnDeviceSizeInBytes;
  mock_api.PJRT_Buffer_Device = Buffer_Device;
  mock_api.PJRT_Buffer_Memory = Buffer_Memory;
  mock_api.PJRT_Buffer_Delete = Buffer_Delete;
  mock_api.PJRT_Buffer_IsDeleted = Buffer_IsDeleted;
  mock_api.PJRT_Buffer_IsOnCpu = Buffer_IsOnCpu;
  mock_api.PJRT_Buffer_ReadyEvent = Buffer_ReadyEvent;
  mock_api.PJRT_Buffer_ToHostBuffer = Buffer_ToHostBuffer;
  mock_api.PJRT_Buffer_CopyToDevice = Buffer_CopyToDevice;
  mock_api.PJRT_Buffer_CopyToMemory = Buffer_CopyToMemory;

  mock_api.PJRT_LoadedExecutable_Destroy = LoadedExecutable_Destroy;
  mock_api.PJRT_LoadedExecutable_GetExecutable = LoadedExecutable_GetExecutable;
  mock_api.PJRT_LoadedExecutable_AddressableDevices =
      LoadedExecutable_AddressableDevices;
  mock_api.PJRT_LoadedExecutable_Delete = LoadedExecutable_Delete;
  mock_api.PJRT_LoadedExecutable_IsDeleted = LoadedExecutable_IsDeleted;
  mock_api.PJRT_LoadedExecutable_Execute = LoadedExecutable_Execute;

  return &mock_api;
}
'''
open("mock_pjrt.c", "w").write(mock_pjrt_c)
print(len(mock_pjrt_c.splitlines()), "lines of C")

In [ ]:
!cc -shared -fPIC -I include mock_pjrt.c -o mock_pjrt.so && ls -la mock_pjrt.so

## Registration, by hand

When you `pip install jax[cuda]`, a Python entry point tells `xla_bridge` where the plugin's shared library lives, and discovery is automatic. `register_plugin` is that mechanism called directly, no packaging required. `JAX_PLATFORMS=mock` then makes our platform the one JAX selects, the same variable chapter 12 used to pick between real backends.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "mock"

from jax._src import xla_bridge as xb
xb.register_plugin("mock", library_path=os.path.abspath("mock_pjrt.so"))

import jax
devices = jax.devices()
print("devices:", devices)

d = devices[0]
print(d.platform, d.device_kind)      # chapter 1's probe, now answered by your C
print(type(d.client).__name__)

Every string in that output came from a function you just compiled: `platform` from `PJRT_Client_PlatformName`, `device_kind` from `PJRT_DeviceDescription_Kind`, the repr from `PJRT_DeviceDescription_DebugString`. JAX asked your table; your table answered.

**your prediction:** `.lower()` worked before we had any backend at all, back in the JAX track. Will `.compile()` succeed against a plugin whose compiler is the identity function? What has to come back for JAX to be satisfied?

In [ ]:
import jax.numpy as jnp

def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v

spec = jax.ShapeDtypeStruct((64, 64), jnp.float32)
lowered = jax.jit(attend).lower(spec, spec, spec)
compiled = lowered.compile()
print("compiled:", type(compiled).__name__)

# The "optimized program" is the bytes our Compile stored, handed back.
print(compiled.as_text()[:160])

Two things happened, and separating them is most of chapter 14. Lowering ran entirely in JAX: trace to jaxpr, jaxpr to StableHLO, no plugin involved. Then `.compile()` crossed the seam: JAX serialized the module, `PjRtCApiClient` packed it into `PJRT_Client_Compile_Args`, and your C function received the bytes and said yes.

The `as_text()` output is the confession. A real backend would return its optimized HLO there, after the two hundred passes of chapter 4. Ours returns the input program unchanged, and the framework obligingly converts our echoed StableHLO to HLO text for display. No pass ran. Nothing was fused, laid out, or assigned a buffer. Compilation was one `memcpy`.

**your prediction:** `jax.device_put` moves a numpy array to a device. Where do the bytes physically end up on this one?

In [ ]:
import numpy as np

x = jax.device_put(np.arange(12, dtype=np.float32).reshape(3, 4))
print("on device:", x.shape, x.dtype, x.devices())
print("read back:", np.array(x)[2])

The bytes went into a `malloc` inside your plugin, because `PJRT_Client_BufferFromHostBuffer` is one `memcpy` in, and `np.array(x)` is `PJRT_Buffer_ToHostBuffer`, one `memcpy` out. The buffer honestly reports `is_on_cpu = false`. Nothing in the interface can tell the difference between this and HBM on a real accelerator, which is the point: `PjRtBuffer` is a contract about ownership and lifetime, not about silicon.

Now the promise we declined.

In [ ]:
one = np.ones((64, 64), np.float32)
try:
    compiled(one, one, one)
except Exception as e:
    print(type(e).__name__, "\n", str(e)[:200])

In [ ]:
# Even this needs Execute: filling an array with ones is a compiled
# program on this backend, dispatched like any other.
try:
    jnp.ones((4,))
except Exception as e:
    print(type(e).__name__, "\n", str(e)[:200])

The second cell is the sharper lesson. `jnp.ones((4,))` is not a memory operation; it is a tiny program, lowered, compiled, and dispatched through `PJRT_LoadedExecutable_Execute` like the attention block was. On a mock that refuses to execute, almost nothing in `jax.numpy` survives, and the few things that do (`device_put`, `np.array`, `.lower()`, `.compile()`) are exactly the calls that map to interface functions you implemented. The boundary between "JAX the tracer" and "the runtime behind PJRT" is not where most people guess it is.

## mark it run

Chapter 14 (kernels.rudrite.com/xla/interfaces) is the reading this lab makes concrete, and the capstone's Project A is this lab continued: replace the refusal with a real interpreter, or wire `Compile` to something that actually optimizes, and you are no longer mocking.

Provenance: verified 2026-08-10 against jax 0.4.38 / jaxlib 0.4.38 with the header at XLA commit `20a4825` (C API 0.58), on CPU (macOS arm64; the same pinned wheels ship for Colab's linux x86_64). Every output above reproduces by running this notebook top to bottom.